# Multi-Agent Integration Adding Math

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

# Verify it loaded
print("API key set:", "Yes" if os.getenv('NVIDIA_API_KEY') else "No")

API key set: Yes


In [2]:
%%capture
# Install the climate analyzer package
!cd climate_analyzer && pip install -e . && cd ..

## Solution: LangGraph Calculator Agent
Using a pre-built, battle-tested calculator instead of building one from scratch, and use LangGraph to integrate it. 

In [3]:
# Import and visualize the calculator agent structure
import sys
sys.path.append('calculator_agent')

In [4]:
import calculator_agent
print(calculator_agent.__doc__)


Multi-Step Calculator Agent using LangGraph

A general-purpose mathematical calculation agent that can handle complex, multi-step calculations
with full transparency into the calculation process. The agent breaks down complex problems into
steps and uses appropriate mathematical tools to solve them.

AVAILABLE MATHEMATICAL TOOLS:
-----------------------------

1. basic_math(expression: str) -> float
   Description: Evaluate basic mathematical expressions
   Supports: +, -, *, /, ** (power), and parentheses
   Example: basic_math('(42.5 - 38.2) * 2.1')
   Use for: Simple arithmetic, order of operations, basic calculations

2. percentage_change(old_value: float, new_value: float) -> float
   Description: Calculate the percentage change between two values
   Formula: ((new_value - old_value) / old_value) * 100
   Example: percentage_change(old_value=35.2, new_value=28.7)
   Use for: Growth rates, comparing changes, trend analysis

3. compound_growth_rate(initial_value: float, final_value

### Test Complex Calculation
Test the calculator agent in isolation before integrating it:

In [5]:
# Test a complex multi-step problem
complex_question = """\
A country's emissions were 1,200 Mt in 2015. They reduced emissions by 2.5% annually until 2020, \
then accelerated reductions to 4% annually. What are the emissions in 2025?"""

In [6]:
from calculator_agent import calculate
result = calculate(complex_question)
print(f"\nFinal Answer: {result['explanation']}")

/usr/local/lib/python3.11/site-packages/langchain_nvidia_ai_endpoints/_common.py:171: UserWarning: http://jupyter-api-proxy.internal.dlai/rev-proxy/nvidia does not end in /v1, you may have inference and listing issues. This check will be deprecated in the next release. Please ensure /v1 is appended to the provided URL
  warnings.warn(



Final Answer: The emissions in 2025 are approximately 883.86 Mt.


## See Current Workflow Fail

In [7]:
# Variant of complex question above using a specific country from our data
complex_nat_question = """\
Get the temperature statistics for India and find its trend per decade. If India's temperature \
continues to increase at this rate, what will the temperature be in 2050?"""

In [8]:
!cd climate_analyzer && nat run --config_file src/climate_analyzer/configs/config.yml --input "$complex_nat_question"

2026-04-18 06:56:02 - INFO     - matplotlib.font_manager:1639 - generated new fontManager
2026-04-18 06:56:04 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'src/climate_analyzer/configs/config.yml'
2026-04-18 06:56:04 - WARNING  - nat.builder.function_info:455 - Using provided input_schema for multi-argument function
2026-04-18 06:56:04 - WARNING  - nat.builder.function_info:455 - Using provided input_schema for multi-argument function
/usr/local/lib/python3.11/site-packages/langchain_nvidia_ai_endpoints/_common.py:171: UserWarning: http://jupyter-api-proxy.internal.dlai/rev-proxy/nvidia does not end in /v1, you may have inference and listing issues. This check will be deprecated in the next release. Please ensure /v1 is appended to the provided URL
  warnings.warn(

Configuration Summary:
--------------------
Workflow Type: react_agent
Number of Functions: 6
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of O

```python
# Import the necessary framework enum for LangChain/LangGraph
from nat.builder.framework_enum import LLMFrameworkEnum

# Register the calculator agent with NAT, specifying it requires LangChain framework for LLM interactions
@register_function(config_type=CalculatorAgentConfig, framework_wrappers=[LLMFrameworkEnum.LANGCHAIN])
async def calculator_agent_tool(config: CalculatorAgentConfig, builder: Builder):
    """Register the LangGraph calculator agent as a NAT tool."""
    
    # Get the LLM from NAT's builder (the config file)
    # The "calculator_llm" refers to the LLM defined in config.yml
    llm = await builder.get_llm("calculator_llm", wrapper_type=LLMFrameworkEnum.LANGCHAIN)
    
    calculator_agent = create_calculator_agent(llm)
    
    async def _wrapper(question: str) -> str:
        result = calculate_with_agent(question, calculator_agent)
        
        response = {
            "calculation_steps": result["steps"],
            "final_result": result["final_result"],
            "explanation": result["explanation"]
        }
        return json.dumps(response, indent=2)
    
    yield FunctionInfo.from_fn(
        _wrapper,
        input_schema=CalculatorInput,
        description=(
            "Calculates compound growth rates, percentage changes, weighted averages, "
            "projections, and multi-step calculations. Shows all calculation steps. "
            "Does not have access to climate data. If calculations need to be performed on climate data, "
            "be sure to acquire that data with other tools before including it in the input question to this tool."
        )
    )
```

<!-- #endregion -->


### Configure Two LLMs
Now config has been updated to define both LLMs and register the calculator agent:

```yaml
llms:
  # Existing climate LLM for general analysis
  climate_llm:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    base_url: https://integrate.api.nvidia.com/v1
    api_key: $NVIDIA_API_KEY
    temperature: 0.0
    top_p: 0.95
    max_tokens: 2048
  
  # New: Dedicated LLM for the calculator agent
  calculator_llm:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    base_url: https://integrate.api.nvidia.com/v1
    api_key: $NVIDIA_API_KEY
    temperature: 0.0  # Low temperature for consistent math
    max_tokens: 1024  # Sufficient for calculations

functions:
  # ... existing functions ...
  
  # New: Register the calculator agent as a function
  calculator_agent:
    _type: climate_analyzer/calculator_agent
    description: "Perform complex mathematical calculations for climate data analysis"

workflow:
  _type: react_agent
  tool_names:
    - list_countries
    - calculate_statistics
    - filter_by_country
    - find_extreme_years
    - create_visualization
    - station_statistics
    - calculator_agent  # Add to available tools
  llm_name: climate_llm
  verbose: true
  max_iterations: 5
  parse_agent_response_max_retries: 2
  max_tool_calls: 30
```


In [9]:
!cd climate_analyzer && nat run --config_file src/climate_analyzer/configs/config_calculator.yml --input "$complex_nat_question"

2026-04-18 06:56:39 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'src/climate_analyzer/configs/config_calculator.yml'
2026-04-18 06:56:39 - WARNING  - nat.builder.function_info:455 - Using provided input_schema for multi-argument function
2026-04-18 06:56:39 - WARNING  - nat.builder.function_info:455 - Using provided input_schema for multi-argument function
/usr/local/lib/python3.11/site-packages/langchain_nvidia_ai_endpoints/_common.py:171: UserWarning: http://jupyter-api-proxy.internal.dlai/rev-proxy/nvidia does not end in /v1, you may have inference and listing issues. This check will be deprecated in the next release. Please ensure /v1 is appended to the provided URL
  warnings.warn(
/usr/local/lib/python3.11/site-packages/langchain_nvidia_ai_endpoints/_common.py:171: UserWarning: http://jupyter-api-proxy.internal.dlai/rev-proxy/nvidia does not end in /v1, you may have inference and listing issues. This check will be deprecated in the next release. Pleas

2026-04-18 07:04:06 - INFO     - nat.agent.react_agent.agent:193 - 
------------------------------
[AGENT]
Agent input: Get the temperature statistics for India and find its trend per decade. If India's temperature continues to increase at this rate, what will the temperature be in 2050?
Agent's thoughts: 
Thought: The calculator agent tool call failed due to a gateway timeout. I will try to simplify the question and try again. Action: calculator_agent Action Input: {"question": "If the temperature increases by 0.154 degrees per decade, how much will it increase in 2.5 decades (from 2025 to 2050)?"}
------------------------------
2026-04-18 07:04:11 - INFO     - nat.agent.base:221 - 
------------------------------
[AGENT]
Calling tools: calculator_agent
Tool's input: {'question': 'If the temperature increases by 0.154 degrees per decade, how much will it increase in 2.5 decades (from 2025 to 2050)?'}
Tool's response: 
{
  "calculation_steps": [
    "basic_math(expression=0.154 * 2.5)"


## Clean Up

In [10]:
!pip uninstall climate_analyzer -y

Found existing installation: climate_analyzer 0.1.0
Uninstalling climate_analyzer-0.1.0:
  Successfully uninstalled climate_analyzer-0.1.0
